In [1]:
import random
import numpy as np
import os
import torch as torch
from src.load_data import load_EOD_data
from src.evaluator import evaluate
from src.model import get_loss, StockMixer
import pickle
from utils import set_seed


device = torch.device("cuda") if torch.cuda.is_available() else 'cpu'
# device = 'cpu'

data_path = './dataset'
# data_path = '../dataset'
market_name = 'NASDAQ'
relation_name = 'wikidata'
stock_num = 1026
lookback_length = 16
epochs = 100
valid_index = 756
test_index = 1008
fea_num = 5
market_num = 20
steps = 1
learning_rate = 0.001
alpha = 0.1
scale_factor = 3
activation = 'GELU'



def validate(start_index, end_index):
    with torch.no_grad():
        cur_valid_pred = np.zeros([stock_num, end_index - start_index], dtype=float)
        cur_valid_gt = np.zeros([stock_num, end_index - start_index], dtype=float)
        cur_valid_mask = np.zeros([stock_num, end_index - start_index], dtype=float)
        loss = 0.
        reg_loss = 0.
        rank_loss = 0.
        for cur_offset in range(start_index - lookback_length - steps + 1, end_index - lookback_length - steps + 1):
            data_batch, mask_batch, price_batch, gt_batch = map(

                lambda x: torch.Tensor(x).to(device),
                get_batch(cur_offset)
            )
            prediction = model(data_batch)
            cur_loss, cur_reg_loss, cur_rank_loss, cur_rr = get_loss(prediction, gt_batch, price_batch, mask_batch,
                                                                     stock_num, alpha)
            loss += cur_loss.item()
            reg_loss += cur_reg_loss.item()
            rank_loss += cur_rank_loss.item()
            cur_valid_pred[:, cur_offset - (start_index - lookback_length - steps + 1)] = cur_rr[:, 0].cpu()
            cur_valid_gt[:, cur_offset - (start_index - lookback_length - steps + 1)] = gt_batch[:, 0].cpu()
            cur_valid_mask[:, cur_offset - (start_index - lookback_length - steps + 1)] = mask_batch[:, 0].cpu()
        loss = loss / (end_index - start_index)
        reg_loss = reg_loss / (end_index - start_index)
        rank_loss = rank_loss / (end_index - start_index)
        cur_valid_perf = evaluate(cur_valid_pred, cur_valid_gt, cur_valid_mask)
    return loss, reg_loss, rank_loss, cur_valid_perf


def get_batch(offset=None):
    if offset is None:
        offset = random.randrange(0, valid_index)
    seq_len = lookback_length
    mask_batch = mask_data[:, offset: offset + seq_len + steps]
    mask_batch = np.min(mask_batch, axis=1)
    return (
        eod_data[:, offset:offset + seq_len, :],
        np.expand_dims(mask_batch, axis=1),
        np.expand_dims(price_data[:, offset + seq_len - 1], axis=1),
        np.expand_dims(gt_data[:, offset + seq_len + steps - 1], axis=1))

In [ ]:
seeds = [2023, 2024, 2025, 2026, 2027]
best_test_perf_lst = []

for seed in seeds:
    set_seed(seed)

    dataset_path = './dataset/' + market_name
    if market_name == "SP500":
        data = np.load('./dataset/SP500/SP500.npy')
        data = data[:, 915:, :]
        price_data = data[:, :, -1]
        mask_data = np.ones((data.shape[0], data.shape[1]))
        eod_data = data
        gt_data = np.zeros((data.shape[0], data.shape[1]))
        for ticket in range(0, data.shape[0]):
            for row in range(1, data.shape[1]):
                gt_data[ticket][row] = (data[ticket][row][-1] - data[ticket][row - steps][-1]) / \
                                    data[ticket][row - steps][-1]
    else:
        with open(os.path.join(dataset_path, "eod_data.pkl"), "rb") as f:
            eod_data = pickle.load(f)
        with open(os.path.join(dataset_path, "mask_data.pkl"), "rb") as f:
            mask_data = pickle.load(f)
        with open(os.path.join(dataset_path, "gt_data.pkl"), "rb") as f:
            gt_data = pickle.load(f)
        with open(os.path.join(dataset_path, "price_data.pkl"), "rb") as f:
            price_data = pickle.load(f)

    trade_dates = mask_data.shape[1]
    model = StockMixer(
        stocks=stock_num,
        time_steps=lookback_length,
        channels=fea_num,
        market=market_num,
        scale=scale_factor
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = np.inf
    best_valid_perf = None
    best_test_perf = None
    batch_offsets = np.arange(start=0, stop=valid_index, dtype=int)


    for epoch in range(epochs):
        print("epoch{}##########################################################".format(epoch + 1))
        np.random.shuffle(batch_offsets)
        tra_loss = 0.0
        tra_reg_loss = 0.0
        tra_rank_loss = 0.0
        for j in range(valid_index - lookback_length - steps + 1):
            data_batch, mask_batch, price_batch, gt_batch = map(
                lambda x: torch.Tensor(x).to(device),
                get_batch(batch_offsets[j])
            )
            optimizer.zero_grad()
            prediction = model(data_batch)
            cur_loss, cur_reg_loss, cur_rank_loss, _ = get_loss(prediction, gt_batch, price_batch, mask_batch,
                                                                stock_num, alpha)
            cur_loss = cur_loss
            cur_loss.backward()
            optimizer.step()

            tra_loss += cur_loss.item()
            tra_reg_loss += cur_reg_loss.item()
            tra_rank_loss += cur_rank_loss.item()
        tra_loss = tra_loss / (valid_index - lookback_length - steps + 1)
        tra_reg_loss = tra_reg_loss / (valid_index - lookback_length - steps + 1)
        tra_rank_loss = tra_rank_loss / (valid_index - lookback_length - steps + 1)
        print('Train : loss:{:.2e}  =  {:.2e} + alpha*{:.2e}'.format(tra_loss, tra_reg_loss, tra_rank_loss))

        val_loss, val_reg_loss, val_rank_loss, val_perf = validate(valid_index, test_index)
        print('Valid : loss:{:.2e}  =  {:.2e} + alpha*{:.2e}'.format(val_loss, val_reg_loss, val_rank_loss))

        test_loss, test_reg_loss, test_rank_loss, test_perf = validate(test_index, trade_dates)
        print('Test: loss:{:.2e}  =  {:.2e} + alpha*{:.2e}'.format(test_loss, test_reg_loss, test_rank_loss))

        if val_loss < best_valid_loss:
            best_valid_loss = val_loss
            best_valid_perf = val_perf
            best_test_perf = test_perf

        print('Valid performance:\n', 'mse:{:.2e}, IC:{:.2e}, RIC:{:.2e}, prec@10:{:.2e}, SR:{:.2e}'.format(val_perf['mse'], val_perf['IC'],
                                                        val_perf['RIC'], val_perf['prec_10'], val_perf['sharpe5']))
        print('Test performance:\n', 'mse:{:.2e}, IC:{:.2e}, RIC:{:.2e}, prec@10:{:.2e}, SR:{:.2e}'.format(test_perf['mse'], test_perf['IC'],
                                                                                test_perf['RIC'], test_perf['prec_10'], test_perf['sharpe5']), '\n\n')
    best_test_perf_lst.append(best_test_perf)


epoch1##########################################################
Train : loss:3.61e-02  =  3.60e-02 + alpha*9.18e-04
Valid : loss:3.87e-03  =  3.82e-03 + alpha*5.05e-04
Test: loss:5.29e-03  =  5.24e-03 + alpha*4.55e-04
Valid performance:
 mse:3.85e-03, IC:2.27e-02, RIC:2.43e-01, prec@10:5.30e-01, SR:1.94e+00
Test performance:
 mse:5.26e-03, IC:3.15e-02, RIC:4.10e-01, prec@10:5.13e-01, SR:2.72e-01 


epoch2##########################################################


In [ ]:
print(best_test_perf_lst)

[{'mse': 0.005256214681147804,
  'IC': 0.031486137821673185,
  'RIC': 0.41006013495123633,
  'sharpe5': 0.2724043171542639,
  'prec_10': 0.5130801687763713},
 {'mse': 0.002613603625430311,
  'IC': 0.029696533577514728,
  'RIC': 0.3748364180568127,
  'sharpe5': 0.842785735804199,
  'prec_10': 0.5244725738396624},
 {'mse': 0.002134987064970432,
  'IC': 0.02811289984466253,
  'RIC': 0.3557367269116282,
  'sharpe5': 1.1331891693922045,
  'prec_10': 0.5210970464135021},
 {'mse': 0.0022950345695871817,
  'IC': 0.026802757754104442,
  'RIC': 0.3294285993234374,
  'sharpe5': 0.513567054660527,
  'prec_10': 0.5122362869198311},
 {'mse': 0.0022950345695871817,
  'IC': 0.026802757754104442,
  'RIC': 0.3294285993234374,
  'sharpe5': 0.513567054660527,
  'prec_10': 0.5122362869198311},
 {'mse': 0.0019926996010530833,
  'IC': 0.02579649860901672,
  'RIC': 0.297877823624765,
  'sharpe5': 0.13737234954754216,
  'prec_10': 0.520675105485232},
 {'mse': 0.0019926996010530833,
  'IC': 0.02579649860901672,